# Heaps / Priority Queue — LeetCode Questions (8 Qs)

Python uses heapq (min-heap by default). For max-heap: negate values.

import heapq
heapq.heappush(heap, val)   # push
heapq.heappop(heap)         # pop smallest
heapq.heapify(list)         # O(n) build heap

1. 215  – Kth Largest Element in an Array
2. 1046 – Last Stone Weight
3. 347  – Top K Frequent Elements
4. 295  – Find Median from Data Stream
5. 23   – Merge K Sorted Lists
6. 973  – K Closest Points to Origin
7. 621  – Task Scheduler
8. 378  – Kth Smallest Element in a Matrix

In [ ]:
import heapq
from typing import List, Optional

# 1. LeetCode 215 – Kth Largest Element in an Array

Find kth largest WITHOUT fully sorting. Use a min-heap of size k.

Input: nums=[3,2,1,5,6,4], k=2  Output: 5

## Dry Run (min-heap of size k)

Keep a min-heap of exactly k largest elements.
The top of heap = kth largest.

| step | element | heap (sorted) | action |
|------|---------|---------------|--------|
| push 3 | 3 | [3] | size=1<2, push |
| push 2 | 2 | [2,3] | size=2 ok |
| push 1 | 1 | [2,3] | 1 < heap[0]=2 → discard 1 |
| push 5 | 5 | [3,5] | 5 > heap[0]=2 → pop 2, push 5 |
| push 6 | 6 | [5,6] | 6 > heap[0]=3 → pop 3, push 6 |
| push 4 | 4 | [5,6] | 4 > heap[0]=3(was) → pop3, push4 → [4,5,6]→size3 pop 4 |
| result | | [5,6] | heap[0] = **5** |

In [ ]:
class Solution:
    def findKthLargest(self, nums: List[int], k: int) -> int:
        min_heap = []
        for num in nums:
            heapq.heappush(min_heap, num)
            if len(min_heap) > k:
                heapq.heappop(min_heap)   # remove smallest, keep k largest
        return min_heap[0]               # top of min-heap = kth largest

print(Solution().findKthLargest([3,2,1,5,6,4], 2))  # 5
print(Solution().findKthLargest([3,2,3,1,2,4,5,5,6], 4))  # 4

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n log k) – push n elements, each push O(log k) | O(k) – heap stores k elements |
| **Final: O(n log k)** | **Final: O(k)** |

> Sorting approach: O(n log n) time, O(1) extra space (but slower)
> Heap approach: O(n log k) — better when k << n

# 2. LeetCode 1046 – Last Stone Weight

Smash two heaviest stones. |y-x| remains if different. Repeat until <=1 stone.

Input: stones=[2,7,4,1,8,1]  Output: 1

Use max-heap (negate for Python min-heap).

## Dry Run

| step | max-heap (descending) | stones smashed | remaining |
|------|----------------------|----------------|-----------|
| start | [8,7,4,2,1,1] | 8,7 → |8-7|=1 | [4,2,1,1,1] |
| 2 | [4,2,1,1,1] | 4,2 → |4-2|=2 | [2,1,1,1] |
| 3 | [2,1,1,1] | 2,1 → |2-1|=1 | [1,1,1] |
| 4 | [1,1,1] | 1,1 → 0 (equal) | [1] |
| done | [1] | 1 stone left | **1** |

In [ ]:
class Solution:
    def lastStoneWeight(self, stones: List[int]) -> int:
        max_heap = [-s for s in stones]   # negate for max-heap
        heapq.heapify(max_heap)

        while len(max_heap) > 1:
            y = -heapq.heappop(max_heap)  # heaviest
            x = -heapq.heappop(max_heap)  # second heaviest
            if y != x:
                heapq.heappush(max_heap, -(y - x))

        return -max_heap[0] if max_heap else 0

print(Solution().lastStoneWeight([2,7,4,1,8,1]))  # 1
print(Solution().lastStoneWeight([1]))             # 1

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n log n) – up to n pops, each O(log n) | O(n) – heap |
| **Final: O(n log n)** | **Final: O(n)** |

# 3. LeetCode 347 – Top K Frequent Elements

Return k most frequent elements (any order).

Input: nums=[1,1,1,2,2,3], k=2  Output: [1,2]

## Dry Run

Step 1: Count frequencies → {1:3, 2:2, 3:1}
Step 2: Use min-heap of size k, keyed by frequency.

| element | freq | heap (freq, elem) | action |
|---------|------|-------------------|--------|
| 1 | 3 | [(3,1)] | push |
| 2 | 2 | [(2,2),(3,1)] | push |
| 3 | 1 | [(2,2),(3,1)] | 1 < heap[0]=2 → discard 3 |
| result | | [(2,2),(3,1)] | return [2,1] → [1,2] |

In [ ]:
from collections import Counter

class Solution:
    def topKFrequent(self, nums: List[int], k: int) -> List[int]:
        count = Counter(nums)               # {1:3, 2:2, 3:1}
        min_heap = []
        for num, freq in count.items():
            heapq.heappush(min_heap, (freq, num))
            if len(min_heap) > k:
                heapq.heappop(min_heap)     # remove least frequent
        return [num for freq, num in min_heap]

print(Solution().topKFrequent([1,1,1,2,2,3], 2))  # [1,2] (any order ok)
print(Solution().topKFrequent([1], 1))             # [1]

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n log k) – count O(n), heap O(n log k) | O(n) – counter + heap |
| **Final: O(n log k)** | **Final: O(n)** |

# 4. LeetCode 295 – Find Median from Data Stream [Hard]

Median: middle value of sorted stream. Design class with addNum + findMedian.

Use TWO heaps: max-heap (left half) + min-heap (right half)
Balance so: len(max_heap) == len(min_heap) OR len(max_heap) == len(min_heap)+1

## Dry Run

addNum(1), addNum(2), addNum(3)

| call | action | max_heap(left) | min_heap(right) | median |
|------|--------|---------------|-----------------|--------|
| add(1) | push to max_heap | [-1] | [] | 1.0 |
| add(2) | push to max, balance | [-1] | [2] | (1+2)/2=1.5 |
| add(3) | push to max, balance | [-2,-1] → [-2]? | | |
| Actually: | push 3 to max, 3>max_top=1, move top to min | [-1] | [2,3] | balance: move min top to max | [-2,-1] → [-1] | [3] | |
| Rebalance | max must have equal or 1 more | [-2] ... | | **2.0** |

In [ ]:
class MedianFinder:
    def __init__(self):
        self.small = []   # max-heap (left half) — negate values
        self.large = []   # min-heap (right half)

    def addNum(self, num: int) -> None:
        # Always push to small first
        heapq.heappush(self.small, -num)
        # Balance: small's max must be <= large's min
        if self.small and self.large and -self.small[0] > self.large[0]:
            heapq.heappush(self.large, -heapq.heappop(self.small))
        # Size balance: small can have at most 1 extra
        if len(self.small) > len(self.large) + 1:
            heapq.heappush(self.large, -heapq.heappop(self.small))
        if len(self.large) > len(self.small):
            heapq.heappush(self.small, -heapq.heappop(self.large))

    def findMedian(self) -> float:
        if len(self.small) > len(self.large):
            return float(-self.small[0])
        return (-self.small[0] + self.large[0]) / 2.0

mf = MedianFinder()
mf.addNum(1); mf.addNum(2)
print(mf.findMedian())  # 1.5
mf.addNum(3)
print(mf.findMedian())  # 2.0

| Operation | Time Complexity | Space Complexity |
|-----------|-----------------|-----------------|
| addNum | O(log n) – heap push/pop | O(n) – store all numbers |
| findMedian | O(1) – peek tops | O(1) |
| **Final** | **O(log n) per add** | **O(n)** |

# 5. LeetCode 23 – Merge K Sorted Lists [Hard]

Merge k sorted linked lists into one sorted list.
Use a min-heap: always extract the minimum head across all k lists.

Input: lists=[[1,4,5],[1,3,4],[2,6]]  Output: [1,1,2,3,4,4,5,6]

## Dry Run

Initial heap (val, list_index, node): [(1,0,L0), (1,1,L1), (2,2,L2)]

| step | pop | add to result | push to heap |
|------|-----|---------------|--------------|
| 1 | (1,0) | 1 | next of L0 = (4,0) |
| 2 | (1,1) | 1 | next of L1 = (3,1) |
| 3 | (2,2) | 2 | next of L2 = (6,2) |
| 4 | (3,1) | 3 | next of L1 = (4,1) |
| 5 | (4,0) | 4 | next of L0 = (5,0) |
| 6 | (4,1) | 4 | L1 exhausted |
| 7 | (5,0) | 5 | L0 exhausted |
| 8 | (6,2) | 6 | L2 exhausted |
| Result: 1→1→2→3→4→4→5→6 | | | |

In [ ]:
from typing import Optional

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

class Solution:
    def mergeKLists(self, lists: List[Optional[ListNode]]) -> Optional[ListNode]:
        dummy = ListNode(0)
        curr = dummy
        heap = []

        for i, node in enumerate(lists):
            if node:
                heapq.heappush(heap, (node.val, i, node))

        while heap:
            val, i, node = heapq.heappop(heap)
            curr.next = node
            curr = curr.next
            if node.next:
                heapq.heappush(heap, (node.next.val, i, node.next))

        return dummy.next

# Build lists: [1→4→5], [1→3→4], [2→6]
def make_list(vals):
    dummy = ListNode(0); curr = dummy
    for v in vals: curr.next = ListNode(v); curr = curr.next
    return dummy.next

lists = [make_list([1,4,5]), make_list([1,3,4]), make_list([2,6])]
result = Solution().mergeKLists(lists)
while result: print(result.val, end=' '); result = result.next
# 1 1 2 3 4 4 5 6

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n log k) – n total nodes, heap size k | O(k) – heap holds k nodes at most |
| **Final: O(n log k)** | **Final: O(k)** |

# 6. LeetCode 973 – K Closest Points to Origin

Return k points closest to origin (0,0). Distance = sqrt(x^2 + y^2).
No need for sqrt — just compare x^2+y^2.

Input: points=[[1,3],[-2,2]], k=1  Output: [[-2,2]]

## Dry Run (max-heap of size k)

Keep k closest: use max-heap, pop farthest when size > k.

| point | dist^2 | heap (dist, point) | action |
|-------|--------|-------------------|--------|
| [1,3] | 10 | [(-10,[1,3])] | push |
| [-2,2] | 8 | [(-10,[1,3]),(-8,[-2,2])] | push |
| size=2=k | | pop max=-10 → remove [1,3] | [(-8,[-2,2])] |
| Result | | [[-2,2]] | |

In [ ]:
class Solution:
    def kClosest(self, points: List[List[int]], k: int) -> List[List[int]]:
        max_heap = []
        for x, y in points:
            dist = x*x + y*y
            heapq.heappush(max_heap, (-dist, x, y))   # negate for max-heap
            if len(max_heap) > k:
                heapq.heappop(max_heap)               # remove farthest

        return [[x, y] for _, x, y in max_heap]

print(Solution().kClosest([[1,3],[-2,2]], 1))   # [[-2,2]]
print(Solution().kClosest([[3,3],[5,-1],[-2,4]], 2))  # two closest

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n log k) – push n points, heap size k | O(k) – heap |
| **Final: O(n log k)** | **Final: O(k)** |

# 7. LeetCode 621 – Task Scheduler

Given task list and cooldown n, find minimum intervals to finish all tasks.
Same task must be at least n intervals apart. Insert idle if needed.

Input: tasks=['A','A','A','B','B','B'], n=2  Output: 8

## Dry Run

Most frequent task = A (3 times), B (3 times), n=2

Strategy: arrange most frequent first, fill gaps with others or idle.

| slot | task | reasoning |
|------|------|-----------|
| 1 | A | most frequent |
| 2 | B | second frequent |
| 3 | idle | nothing available |
| 4 | A | cooldown over |
| 5 | B | cooldown over |
| 6 | idle | |
| 7 | A | |
| 8 | B | |
| Result: 8 intervals | | |

Formula: max((max_count-1)*(n+1) + tasks_with_max_count, total_tasks)
= (3-1)*(2+1) + 2 = 8, total=6 → max(8,6) = 8

In [ ]:
from collections import Counter

class Solution:
    def leastInterval(self, tasks: List[str], n: int) -> int:
        count = Counter(tasks)
        max_count = max(count.values())
        tasks_with_max = sum(1 for c in count.values() if c == max_count)
        result = (max_count - 1) * (n + 1) + tasks_with_max
        return max(result, len(tasks))   # can't be less than total tasks

print(Solution().leastInterval(['A','A','A','B','B','B'], 2))   # 8
print(Solution().leastInterval(['A','A','A','B','B','B'], 0))   # 6

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(n) – count tasks once | O(1) – at most 26 distinct tasks |
| **Final: O(n)** | **Final: O(1)** |

# 8. LeetCode 378 – Kth Smallest Element in a Sorted Matrix

Matrix rows AND columns sorted. Find kth smallest element.
Use min-heap starting from top-left, expand to right and down.

Input: matrix=[[1,5,9],[10,11,13],[12,13,15]], k=8  Output: 13

## Dry Run (min-heap)

Start with matrix[0][0]=1. Each pop, push right and down neighbors.

| step | pop (val,r,c) | push neighbors | heap top |
|------|--------------|----------------|----------|
| 1 | (1,0,0) | (5,0,1),(10,1,0) | 5 |
| 2 | (5,0,1) | (9,0,2),(11,1,1) | 9 |
| 3 | (9,0,2) | (13,1,2) | 10 |
| 4 | (10,1,0) | (12,2,0) | 11 |
| 5 | (11,1,1) | (13,2,1) | 12 |
| 6 | (12,2,0) | (13,2,1)dup | 13 |
| 7 | (13,1,2) | (15,2,2) | 13 |
| 8 | **(13,2,1)** | k=8, **return 13** | |

In [ ]:
class Solution:
    def kthSmallest(self, matrix: List[List[int]], k: int) -> int:
        n = len(matrix)
        heap = [(matrix[0][0], 0, 0)]
        visited = {(0, 0)}

        for _ in range(k - 1):
            val, r, c = heapq.heappop(heap)
            if r + 1 < n and (r+1, c) not in visited:
                heapq.heappush(heap, (matrix[r+1][c], r+1, c))
                visited.add((r+1, c))
            if c + 1 < n and (r, c+1) not in visited:
                heapq.heappush(heap, (matrix[r][c+1], r, c+1))
                visited.add((r, c+1))

        return heapq.heappop(heap)[0]

matrix = [[1,5,9],[10,11,13],[12,13,15]]
print(Solution().kthSmallest(matrix, 8))   # 13

| Time Complexity | Space Complexity |
|-----------------|-----------------|
| O(k log k) – k pops, heap size at most k | O(k) – heap + visited set |
| **Final: O(k log k)** | **Final: O(k)** |